# 🎬 Otaku AI Viral Shorts Studio
### Single-Cell Edition — Google Colab & Kaggle

**How to run:**
1. Add your secrets in **Colab → Secrets (🔑)** (left sidebar):
   - `GEMINI_API_KEY` → your Gemini 2.5 Flash API key (from [ai.google.dev](https://ai.google.dev))
   - `YOUTUBE_COOKIES_TXT` → paste your full `cookies.txt` content (Netscape format) — optional, but fixes YouTube bot-check errors
2. Set parameters in the form below (or leave defaults)
3. Click **▶ Run** — it handles everything: install, clone, harvest (IG + TikTok + YouTube), AI frame inspection, 1080×1920 render, QA, metadata

> **Pillar Days:** 1=Cinema | 2=TV Series | 3=The Mentalist (LOCKED) | 4=Song Trend Edits | 0=Auto (follows rotating wheel)

In [ ]:
#@title 🎬 Otaku AI Viral Shorts Studio — All-in-One Single Cell
RUN_FULL_CYCLE = True #@param {type:"boolean"}
RUN_EVERY_8H = False #@param {type:"boolean"}
MAX_8H_CYCLES = 3 #@param {type:"integer"}
AUTO_DISCONNECT_GPU = False #@param {type:"boolean"}
TOPIC_OVERRIDE = "" #@param {type:"string"}
PILLAR_DAY_OVERRIDE = 0 #@param [0, 1, 2, 3, 4] {type:"raw"}
USE_LOCAL_SMALL_MODEL = False #@param {type:"boolean"}
DURATION_MODE = "AI Decides Cut Duration (Full Climax)" #@param ["AI Decides Cut Duration (Full Climax)", "Full Uncut (Patrick Jane Style)", "Custom Seconds"]
CUSTOM_SECONDS = 18 #@param {type:"integer"}
UPLOAD_TO_YOUTUBE = False #@param {type:"boolean"}
GEMINI_API_KEY = "" #@param {type:"string"}

# ============================================================
# PHASE 1: ENVIRONMENT SETUP
# ============================================================
import os, sys, subprocess, shutil, importlib, pathlib, time
print("⚙️  [1/5] Setting up cloud environment...")

# 1a. Suppress transformers trust_remote_code interactive prompts globally
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"
os.environ["TRANSFORMERS_TRUST_REMOTE_CODE"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
# Silence pip r2u/ubuntu repo warnings
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

# 1b. System packages (ffmpeg + fonts) — quiet, only if missing
if not shutil.which("ffmpeg"):
    print("   Installing ffmpeg & fonts...")
    subprocess.run(
        "apt-get update -qq 2>/dev/null && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation >/dev/null 2>&1",
        shell=True, check=False
    )

# 1b-2. Ollama local vision model (when USE_LOCAL_SMALL_MODEL is True)
if USE_LOCAL_SMALL_MODEL:
    _ollama_bin = shutil.which("ollama") or ("/usr/local/bin/ollama" if os.path.exists("/usr/local/bin/ollama") else None)
    if not _ollama_bin:
        print("   Installing Ollama...")
        subprocess.run("apt-get install -y -qq zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1", shell=True, check=False)
        _ollama_bin = shutil.which("ollama") or ("/usr/local/bin/ollama" if os.path.exists("/usr/local/bin/ollama") else None)
    if _ollama_bin:
        subprocess.Popen([_ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        print("   Pulling Ollama vision model (qwen2.5vl:3b)...")
        subprocess.run([_ollama_bin, "pull", "qwen2.5vl:3b"], check=False)
    else:
        print("   ⚠️  Ollama install failed; will fall back to Gemini API.")

# 1c. Python packages — install quietly
print("   Installing Python packages...")
_pkgs = [
    "yt-dlp[default]",
    "curl_cffi",
    "pillow",
    "opencv-python-headless",
    "google-generativeai",
    "google-api-python-client",
    "google-auth-oauthlib",
    "numpy",
    "torch",
    "torchvision",
    "transformers>=4.45",
    "accelerate",
    "qwen-vl-utils",
    "git+https://github.com/Panniantong/Agent-Reach.git",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"] + _pkgs,
    stdout=subprocess.DEVNULL, check=False
)

# ============================================================
# PHASE 2: REPO CLONE / PULL
# ============================================================
print("📦 [2/5] Cloning / updating studio repo...")
_REPO_URL = "https://github.com/sp25126/MOVIE-SERIES---AUTO.git"
_REPO_DIR = "/content/otaku_repo"

if os.path.isfile("autonomous_pipeline.py"):
    # Already in the repo directory — force sync to latest origin/main (keeps untracked cookies/.env)
    subprocess.run(["git", "fetch", "-q", "origin", "main"], check=False)
    subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], check=False)
else:
    if not os.path.isdir(_REPO_DIR):
        subprocess.run(["git", "clone", "-q", _REPO_URL, _REPO_DIR], check=True)
    os.chdir(_REPO_DIR)
    subprocess.run(["git", "fetch", "-q", "origin", "main"], check=False)
    subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], check=False)

_cwd = os.getcwd()
if _cwd not in sys.path:
    sys.path.insert(0, _cwd)
print(f"   Repo ready at: {_cwd}")

# ============================================================
# PHASE 3: LOAD SECRETS
# ============================================================
print("🔑 [3/5] Loading secrets...")
os.makedirs("config", exist_ok=True)

# Gemini API Key
if not GEMINI_API_KEY.strip():
    try:
        from google.colab import userdata as _ud
        GEMINI_API_KEY = (_ud.get("GEMINI_API_KEY") or "").strip()
    except Exception:
        pass
if not GEMINI_API_KEY.strip():
    # Try .env or Google Drive
    for _env_p in [pathlib.Path(".env"), pathlib.Path("/content/drive/MyDrive/.env")]:
        if _env_p.exists():
            for _line in _env_p.read_text().splitlines():
                if _line.strip().startswith("GEMINI_API_KEY="):
                    GEMINI_API_KEY = _line.split("=", 1)[1].strip().strip('"').strip("'")
                    break
        if GEMINI_API_KEY.strip():
            break
if GEMINI_API_KEY.strip():
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY.strip()
    print(f"   ✅ Gemini API key loaded ({len(GEMINI_API_KEY)} chars)")
else:
    print("   ⚠️  No GEMINI_API_KEY found — add it in Colab Secrets (🔑) as 'GEMINI_API_KEY'")

# YouTube cookies — from Colab Secret > Google Drive > local file
_ck_path = pathlib.Path("config/cookies.txt")
if not (_ck_path.exists() and _ck_path.stat().st_size > 100):
    try:
        from google.colab import userdata as _ud
        _ck_txt = _ud.get("YOUTUBE_COOKIES_TXT") or ""
        if _ck_txt.strip():
            _ck_path.write_text(_ck_txt, encoding="utf-8")
            print("   ✅ YouTube cookies loaded from Colab Secret.")
    except Exception:
        pass
if not (_ck_path.exists() and _ck_path.stat().st_size > 100):
    for _gdrive_ck in ["/content/drive/MyDrive/cookies.txt", "/content/drive/MyDrive/youtube_cookies.txt"]:
        if os.path.exists(_gdrive_ck):
            shutil.copy(_gdrive_ck, str(_ck_path))
            print(f"   ✅ YouTube cookies loaded from Google Drive ({_gdrive_ck}).")
            break
if not (_ck_path.exists() and _ck_path.stat().st_size > 100):
    print("   ℹ️  No cookies.txt found — YouTube downloads may hit bot-check errors.")
    print("       To fix: add YOUTUBE_COOKIES_TXT to Colab Secrets (🔑) with your cookies.txt content.")

# ============================================================
# PHASE 4: RELOAD MODULES (picks up latest git pull changes)
# ============================================================
print("🔄 [4/5] Reloading pipeline modules...")
import category_skills, proxy_harvester, clip_harvester, channel_intelligence, llm_director, dsp_renderer
import metadata_uploader, autonomous_pipeline
_MODULES_TO_RELOAD = [category_skills, proxy_harvester, clip_harvester, channel_intelligence, llm_director, dsp_renderer, metadata_uploader, autonomous_pipeline]
for _opt_mod in ("virality_radar", "tv_series_radar", "song_trend_radar", "virality_engine", "llm_virality_brain", "song_ranking_renderer", "qa_gate"):
    try:
        _MODULES_TO_RELOAD.append(importlib.import_module(_opt_mod))
    except ImportError:
        pass
for _m in _MODULES_TO_RELOAD:
    try:
        importlib.reload(_m)
    except Exception as _e:
        print(f"   ⚠️  Reload warning ({_m.__name__}): {_e}")

# ============================================================
# PHASE 5: RUN 8-STEP PIPELINE (SINGLE OR 8-HOUR AUTO LOOP)
# ============================================================
CLIP_DURATION_SEC = 0.0 if ("AI Decides" in DURATION_MODE or "Full Uncut" in DURATION_MODE) else float(CUSTOM_SECONDS)
_total_cycles = int(MAX_8H_CYCLES) if RUN_EVERY_8H else 1

for _cycle_idx in range(1, _total_cycles + 1):
    print(f"\n🚀 [5/5] Launching Cycle #{_cycle_idx}/{_total_cycles} (Instagram + TikTok + YouTube)...")
    print(f"   Pillar: {'AUTO' if not PILLAR_DAY_OVERRIDE else PILLAR_DAY_OVERRIDE} | Topic: '{TOPIC_OVERRIDE or 'AUTO'}' | LocalVLM: {USE_LOCAL_SMALL_MODEL}")

    if RUN_FULL_CYCLE and not TOPIC_OVERRIDE.strip():
        _results = autonomous_pipeline.run_full_category_cycle(
            use_local_vlm=bool(USE_LOCAL_SMALL_MODEL),
            upload=bool(UPLOAD_TO_YOUTUBE),
            clip_dur=CLIP_DURATION_SEC,
            start_day=int(PILLAR_DAY_OVERRIDE) if int(PILLAR_DAY_OVERRIDE) in (1, 2, 3, 4) else 1,
        )
    else:
        _results = [autonomous_pipeline.run_autonomous_workflow(
            day_override=int(PILLAR_DAY_OVERRIDE),
            topic_override=TOPIC_OVERRIDE.strip(),
            use_local_vlm=bool(USE_LOCAL_SMALL_MODEL),
            upload=bool(UPLOAD_TO_YOUTUBE),
            clip_dur=CLIP_DURATION_SEC,
        )]

    for result in _results:
        if result.get("status") != "success":
            print(f"\n❌ FAILED Pillar #{result.get('pillar_day')} ({result.get('category')}): {result.get('error')}")
            continue
        out_video = pathlib.Path(result["output_mp4"])
        meta = result["metadata"]
        print(f"\n{'='*60}")
        print(f"🎉 DONE!  {out_video.name}  ({out_video.stat().st_size / 1e6:.2f} MB)")
        print(f"{'='*60}")
        print(f"📅 Pillar:   Day {result['pillar_day']} — {result['category']}")
        print(f"🏆 Topic:    {result['topic']}")
        print(f"📋 Title:    {meta['title']}")
        print(f"⏰ Slot UTC: {meta['recommended_slot_utc']}")
        print(f"🔖 Tags:     {' '.join(meta.get('tags', [])[:10])}")
        print(f"\n--- YOUTUBE DESCRIPTION & CREDITS ---\n")
        print(meta['description'])
        try:
            from IPython.display import Video, display, HTML
            display(HTML(f'<h3>🎬 Preview ({result["category"]}): {result["topic"]}</h3>'))
            display(Video(str(out_video), embed=True, width=360, height=640))
        except Exception:
            print(f"\nVideo saved to: {out_video}")

    if RUN_EVERY_8H and _cycle_idx < _total_cycles:
        print("\n⏳ 8-Hour Scheduler: Waiting 8 hours before next 4-video cycle (keep tab open)...")
        for _rem_min in range(480, 0, -10):
            print(f"   💤 Next cycle in {_rem_min // 60}h {_rem_min % 60:02d}m...")
            time.sleep(600)
        subprocess.run(["git", "fetch", "-q", "origin", "main"], check=False)
        subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], check=False)
        for _m in _MODULES_TO_RELOAD:
            try:
                importlib.reload(_m)
            except Exception:
                pass

if AUTO_DISCONNECT_GPU:
    # Copy outputs to Google Drive if mounted so nothing is lost on runtime unassign
    if os.path.isdir("/content/drive/MyDrive"):
        _drv_out = "/content/drive/MyDrive/Otaku_Shorts_Output"
        os.makedirs(_drv_out, exist_ok=True)
        for _f in pathlib.Path("output").glob("*"):
            if _f.is_file():
                shutil.copy2(str(_f), os.path.join(_drv_out, _f.name))
        print(f"💾 Backed up outputs to Google Drive: {_drv_out}")
    try:
        from google.colab import runtime
        print("🔌 Releasing T4 GPU runtime to save quota...")
        runtime.unassign()
    except Exception:
        pass
